# E1 — So sánh thuật toán tối ưu (yêu cầu 5.1.1)

**Hướng dẫn người chạy:** bật GPU **T4** (không chọn P100) và Internet → sửa `OWNER` → *Save & Run All* (chạy nền) → tải `outputs/` (file `.zip` in ở cuối) gửi cho điều phối. Không sửa gì khác.

Cố định lịch cosine (theo bước), batch 128, 20 epoch, ResNet-18 (width 32) trên CIFAR-10; **chỉ đổi optimizer**. Mỗi cấu hình chạy với seed 0 và 1. Tổng 13 lượt (12 + 1 lượt `E1-sgdm-rerun` chỉ để kiểm tra tái lập).

| config | optimizer | lr | weight decay |
|---|---|---:|---|
| E1-sgd | SGD (momentum 0) | 0.1 | 5e-4 |
| E1-sgdm | SGD momentum 0.9 | 0.1 | 5e-4 |
| E1-nesterov | momentum 0.9 + Nesterov | 0.1 | 5e-4 |
| E1-rmsprop | RMSProp (α 0.99) | 1e-3 | 5e-4 |
| E1-adam | Adam (0.9, 0.999) | 1e-3 | 5e-4 (L2 gộp vào gradient) |
| E1-adamw | AdamW (0.9, 0.999) | 1e-3 | **5e-2** (decoupled) |

## Lý do các lựa chọn chính

- **AdamW dùng weight decay 5e-2:** mỗi bước AdamW co trọng số một lượng `lr × wd`. SGD có `0.1 × 5e-4 = 5e-5`; AdamW `1e-3 × 5e-2 = 5e-5`. Nếu dùng 5e-4 thì gần như không có weight decay và so sánh Adam vs AdamW mất ý nghĩa.
- **Không dò LR riêng cho từng optimizer:** dùng LR mặc định kinh điển (0.1 cho họ SGD, 1e-3 cho RMSProp/Adam/AdamW). **Hạn chế:** chênh lệch có thể lẫn giữa "optimizer" và "LR chưa tối ưu". Nếu RMSProp/Adam phân kỳ ở 1e-3, lượt đó được ghi `diverged` và báo cáo đúng như vậy (không đổi LR sau khi thấy kết quả).
- **Cặp cần đọc:** SGD vs SGDm (tác dụng momentum), SGDm vs Nesterov, Adam vs AdamW (decoupled weight decay), {RMSProp, Adam, AdamW} vs SGDm.
- **`E1-sgdm-rerun`:** chạy lại đúng cấu hình `E1-sgdm` seed 0; kết quả phải trùng đến mọi chữ số (kiểm tra tái lập).

In [ ]:
# ===== CHỈ SỬA CÁC DÒNG TRONG KHỐI NÀY =====
OWNER = "ten_ban"          # tên của bạn (không dấu, không khoảng trắng)
SEEDS = [0, 1]
REPO_URL = "https://github.com/<ORG>/<REPO>.git"   # do điều phối cung cấp
COMMIT = "<hash>"                                  # do điều phối cung cấp
# ============================================

import os, sys
if not os.path.exists("repo"):
    !git clone $REPO_URL repo
!cd repo && git checkout $COMMIT
sys.path.insert(0, "repo")
import pandas as pd
import btl
btl.init(owner=OWNER)

In [ ]:
RUNS = {
    "E1-sgd":      dict(optimizer="sgd"),
    "E1-sgdm":     dict(optimizer="sgdm"),
    "E1-nesterov": dict(optimizer="nesterov"),
    "E1-rmsprop":  dict(optimizer="rmsprop", lr=1e-3),
    "E1-adam":     dict(optimizer="adam",    lr=1e-3),
    "E1-adamw":    dict(optimizer="adamw",   lr=1e-3, weight_decay=5e-2),
}

for seed in SEEDS:                        # seed 0 xong cho mọi cấu hình trước
    for name, overrides in RUNS.items():
        btl.run(name, seed, **overrides)

if 0 in SEEDS:                            # lượt lặp để kiểm tra tái lập (không tính vào mean ± std)
    btl.run("E1-sgdm-rerun", 0, optimizer="sgdm")

In [ ]:
runs = pd.read_csv(f"{btl.OUT_DIR}/runs.csv")
print(runs[["config", "seed", "status", "final_test_acc", "duration_min"]].to_string(index=False))
bad = runs[runs.status != "completed"]
print(f"\nTổng {len(runs)} lượt, {len(bad)} lượt diverged/crashed" + (" -> ĐỌC KỸ cột notes trong runs.csv" if len(bad) else ""))
btl.zip_outputs("01_optimizers")   # gửi file .zip này cho điều phối